# Phase 1 EDA - IBM AMLSim HI-Small Dataset

## SECTION 1: Setup

In [ ]:
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings('ignore')
from src.pipeline.data_ingestion import load_ibm_pipeline, get_summary_stats
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = load_ibm_pipeline('../data/raw/HI-Small_Trans.csv')
print(f"Shape: {df.shape}")
df.head()

## SECTION 2: Dataset Overview

In [ ]:
stats = get_summary_stats(df)
for k, v in stats.items():
    print(f"{k}: {v}")
print("\n--- dtypes ---")
print(df.dtypes)
print("\n--- describe ---")
df.describe()

## SECTION 3: Class Imbalance

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='is_laundering')
total = len(df)
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 3,
            f'{height} ({height/total:.2%})', ha="center")
plt.title("Class Distribution - 99.88% Clean vs 0.12% Laundering")
plt.savefig('../data/reports/eda_class_imbalance.png')
plt.show()

## SECTION 4: Amount Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='amount_log', hue='is_laundering', alpha=0.5, bins=50, common_norm=False, stat="density")
plt.axvline(df[df['is_laundering']==0]['amount_log'].mean(), color='blue', linestyle='--')
plt.axvline(df[df['is_laundering']==1]['amount_log'].mean(), color='orange', linestyle='--')
plt.title("Log-Amount Distribution: Clean vs Laundering")
plt.savefig('../data/reports/eda_amount_distribution.png')
plt.show()

Laundering transactions cluster around amount_log 8-12 (~$3k-$160k), suggesting deliberate mid-range structuring to avoid detection thresholds

## SECTION 5: Transaction Type Analysis

In [ ]:
type_stats = df.groupby('transaction_type')['is_laundering'].agg(['count', 'mean']).reset_index()
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=type_stats, x='transaction_type', y='count', hue='mean', palette='coolwarm')
for idx, p in enumerate(ax.patches):
    if p.get_height() > 0:
        ax.text(p.get_x() + p.get_width()/2., p.get_height(),
                f'{type_stats.iloc[idx % len(type_stats)]["mean"]:.4%}', ha="center")
plt.title("Transaction Type vs Count (colored by laundering rate)")
plt.savefig('../data/reports/eda_transaction_types.png')
plt.show()

## SECTION 6: Temporal Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df.groupby(['hour_of_day', 'is_laundering']).size().reset_index(name='count'),
             x='hour_of_day', y='count', hue='is_laundering')
plt.title("Transaction Count by Hour of Day")
plt.savefig('../data/reports/eda_hourly.png')
plt.show()

Laundering shows no temporal pattern - uniform distribution across all hours. Temporal heuristics alone are insufficient.

## SECTION 7: Cross-Border Analysis

In [ ]:
plt.figure(figsize=(8, 6))
df['is_cross_border_str'] = df['is_cross_border'].map({True: 'Cross-Border', False: 'Domestic'})
sns.histplot(data=df, x='is_cross_border_str', hue='is_laundering', multiple='stack', shrink=0.8)
plt.title("Cross-Border vs Domestic Transactions")
plt.savefig('../data/reports/eda_crossborder.png')
plt.show()

Only 2,191 cross-border transactions (0.05%). Currency used as country proxy in IBM dataset - limited signal for cross-border heuristic.

## SECTION 8: Currency Distribution

In [ ]:
plt.figure(figsize=(10, 6))
df['sender_country'].value_counts().head(10).plot(kind='barh')
plt.title("Top 10 Currencies by Transaction Count")
plt.savefig('../data/reports/eda_currencies.png')
plt.show()

## Key Findings from EDA

1. **Extreme class imbalance** (99.88% clean) makes supervised learning without SMOTE ineffective
2. **Laundering amounts cluster at mid-range** ($3k-$160k) - launderers avoid large amounts to stay below reporting thresholds
3. **No temporal signal** - laundering distributed uniformly across all hours, invalidating time-based heuristics alone
4. **Minimal cross-border activity** (0.05%) - currency proxy is insufficient, graph-based country analysis needed in Phase 2
5. **ACH dominates** (84.9% of transactions) with highest absolute laundering count

These findings directly motivate the hybrid detection approach and graph-based investigation in Phase 2.